In [9]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

In [10]:
text="""
LangChain is a framework used to build applications powered by Large Language Models (LLMs). It provides document loaders, text splitters, retrievers, and prompt templates for creating RAG applications.
LangChain supports integration with different LLM providers and vector databases. Developers can use it to retrieve relevant documents and provide contextual information to language models.
Python is a popular, high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data analysis, and artificial intelligence.
Python supports object-oriented, procedural, and functional programming paradigms. Popular libraries such as Pandas, NumPy, and Scikit-learn help developers process data and build machine learning models.
The Taj Mahal is a famous historical monument located in Agra, India. Built by Mughal emperor Shah Jahan in memory of his wife Mumtaz Mahal, it is renowned for its white marble architecture.
Photosynthesis is the process by which green plants use sunlight, water, and carbon dioxide to produce glucose and release oxygen. This process is essential for sustaining life and maintaining oxygen levels in Earth's atmosphere.
"""

In [11]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
docs = [
    Document(page_content= doc.strip()) for doc in text.splitlines() if doc.strip()
]

retrivier = FAISS.from_documents(docs, embedding=embeddings)
dence_retriver = retrivier.as_retriever()

dence_retriver

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x11dff3d90>, search_kwargs={})

In [15]:
sparse_retriver = BM25Retriever.from_documents(docs)
sparse_retriver.k = 3

In [16]:
from langchain_classic.retrievers import EnsembleRetriever
compbine_retriever = EnsembleRetriever(
    retrievers=[dence_retriver, sparse_retriver],
    weights=[0.5, 0.5]
)


In [17]:
query = "Can you help to understand Python with langchain ?"
result = compbine_retriever.invoke(query)

for i, doc in enumerate(result):
    print(f"Doc {i+1}: {doc.page_content}")

Doc 1: LangChain supports integration with different LLM providers and vector databases. Developers can use it to retrieve relevant documents and provide contextual information to language models.
Doc 2: Python supports object-oriented, procedural, and functional programming paradigms. Popular libraries such as Pandas, NumPy, and Scikit-learn help developers process data and build machine learning models.
Doc 3: Python is a popular, high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data analysis, and artificial intelligence.
Doc 4: LangChain is a framework used to build applications powered by Large Language Models (LLMs). It provides document loaders, text splitters, retrievers, and prompt templates for creating RAG applications.


In [20]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

In [24]:
prompt = ChatPromptTemplate.from_template(
    template=""" Please provide answer of question based on provided context
    Context: {context}
    Question: {input}
    """
)

llm = ChatOpenAI(
    model="gpt-4o-mini"
)

chain_document = create_stuff_documents_chain(llm, prompt)
conv_retrival = create_retrieval_chain(retriever=compbine_retriever, combine_docs_chain=result)

TypeError: Expected a Runnable, callable or dict.Instead got an unsupported type: <class 'list'>